In [3]:
# ============================================================
# Cell 1 — Environment and reproducibility configuration
# ============================================================

from pathlib import Path
import json
import platform
import random
import re
import sys

import numpy as np
import pandas as pd
from datasets import load_dataset


# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/home/chufeng/Desktop/CSI/ICSE2027"
).resolve()

DATA_PREPARATION_DIR = (
    PROJECT_ROOT
    / "Data Preparation"
)

ANNOTATION_DIR = (
    DATA_PREPARATION_DIR
    / "data"
)

NEBIUS_ANNOTATION_FILE = (
    ANNOTATION_DIR
    / "nebius-swe_final.jsonl"
)

HF_CACHE_DIR = (
    PROJECT_ROOT
    / "hf_cache"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "reproduction_results"
    / "rq1_nebius_equal_exposure"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HF_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 2. Dataset and experiment configuration
# ------------------------------------------------------------

NEBIUS_DATASET = "nebius/SWE-agent-trajectories"
NEBIUS_SPLIT = "train"

TP_LABEL = "TEST_PANDERING"

K1_REPLICATIONS = 1_000
K1_RANDOM_SEED = 20260822


# ------------------------------------------------------------
# 3. Canonical model mappings
# ------------------------------------------------------------

NEBIUS_RAW_MODEL_MAP = {
    "swe-agent-llama-405b": "Llama-405B",
    "swe-agent-llama-70b": "Llama-70B",
    "swe-agent-llama-8b": "Llama-8B",
}

NEBIUS_ANNOTATION_MODEL_MAP = {
    "swe-agent-llama-405b": "Llama-405B",
    "swe-agent-llama-70b": "Llama-70B",
    "swe-agent-llama-8b": "Llama-8B",
    "llama-405b": "Llama-405B",
    "llama-70b": "Llama-70B",
    "llama-8b": "Llama-8B",
    "Llama-405B": "Llama-405B",
    "Llama-70B": "Llama-70B",
    "Llama-8B": "Llama-8B",
}


# ------------------------------------------------------------
# 4. Fixed seeds
# ------------------------------------------------------------

random.seed(K1_RANDOM_SEED)
np.random.seed(K1_RANDOM_SEED)


# ------------------------------------------------------------
# 5. Environment report
# ------------------------------------------------------------

environment_summary = pd.DataFrame(
    [
        {
            "Item": "Python",
            "Value": sys.version.split()[0],
        },
        {
            "Item": "Platform",
            "Value": platform.platform(),
        },
        {
            "Item": "pandas",
            "Value": pd.__version__,
        },
        {
            "Item": "NumPy",
            "Value": np.__version__,
        },
        {
            "Item": "HF dataset",
            "Value": NEBIUS_DATASET,
        },
        {
            "Item": "Annotation file",
            "Value": str(NEBIUS_ANNOTATION_FILE),
        },
        {
            "Item": "Random seed",
            "Value": K1_RANDOM_SEED,
        },
        {
            "Item": "Replications",
            "Value": K1_REPLICATIONS,
        },
    ]
)

display(environment_summary)

assert PROJECT_ROOT.exists(), (
    f"Project root does not exist: {PROJECT_ROOT}"
)

assert NEBIUS_ANNOTATION_FILE.is_file(), (
    "Nebius annotation file does not exist:\n"
    f"{NEBIUS_ANNOTATION_FILE}"
)

print("✓ Configuration completed.")
print(f"✓ Results will be saved to: {OUTPUT_DIR}")

,Item,Value
0,Python,3.13.5
1,Platform,Linux-6.18.33.2-microsoft-standard-WSL2-x86_64...
2,pandas,3.0.3
3,NumPy,2.5.1
4,HF dataset,nebius/SWE-agent-trajectories
5,Annotation file,/home/chufeng/Desktop/CSI/ICSE2027/Data Prepar...
6,Random seed,20260822
7,Replications,1000


✓ Configuration completed.
✓ Results will be saved to: /home/chufeng/Desktop/CSI/ICSE2027/reproduction_results/rq1_nebius_equal_exposure


In [4]:
# ============================================================
# Cell 2 — Load and canonicalize Nebius annotations
# ============================================================


def iter_annotation_records(jsonl_path):
    """
    Yield individual annotation records from a JSONL file.

    Supports:
      1. One annotation dictionary per line.
      2. A list of annotation dictionaries per line.
      3. Common wrapper dictionaries containing a list.
    """

    wrapper_keys = [
        "results",
        "annotations",
        "records",
        "items",
        "data",
    ]

    with jsonl_path.open(
        "r",
        encoding="utf-8",
    ) as file:

        for line_number, line in enumerate(
            file,
            start=1,
        ):
            line = line.strip()

            if not line:
                continue

            parsed = json.loads(line)

            if isinstance(parsed, list):
                records = parsed

            elif isinstance(parsed, dict):
                records = None

                for wrapper_key in wrapper_keys:
                    candidate = parsed.get(wrapper_key)

                    if isinstance(candidate, list):
                        records = candidate
                        break

                if records is None:
                    records = [parsed]

            else:
                raise TypeError(
                    "Unsupported annotation JSON type at "
                    f"line {line_number}: "
                    f"{type(parsed).__name__}"
                )

            for record in records:
                if not isinstance(record, dict):
                    raise TypeError(
                        "Annotation record is not a dictionary "
                        f"at line {line_number}."
                    )

                yield record


def canonicalize_annotation_model(record):
    """
    Recover the canonical model name from an annotation record.
    """

    model_candidates = [
        record.get("_canonical_model"),
        record.get("model_name"),
        record.get("model"),
    ]

    for candidate in model_candidates:
        if candidate is None:
            continue

        candidate_string = str(candidate).strip()

        if candidate_string in NEBIUS_ANNOTATION_MODEL_MAP:
            return NEBIUS_ANNOTATION_MODEL_MAP[
                candidate_string
            ]

        candidate_lower = candidate_string.lower()

        if candidate_lower in NEBIUS_ANNOTATION_MODEL_MAP:
            return NEBIUS_ANNOTATION_MODEL_MAP[
                candidate_lower
            ]

    raise ValueError(
        "Cannot canonicalize annotation model:\n"
        f"{record}"
    )


def recover_instance_id(record):
    """
    Recover the PR/instance identifier.

    Nebius trajectory IDs have the form:
        <instance_id>_<zero-based global index>
    """

    explicit_instance_id = record.get("_instance_id")

    if explicit_instance_id:
        return str(explicit_instance_id)

    trajectory_id = record.get("trajectory_id")

    if not trajectory_id:
        raise ValueError(
            "Annotation has neither _instance_id nor "
            "trajectory_id."
        )

    match = re.match(
        r"^(.*)_(\d+)$",
        str(trajectory_id),
    )

    if match is None:
        raise ValueError(
            "Cannot recover instance ID from trajectory ID: "
            f"{trajectory_id}"
        )

    return match.group(1)


# ------------------------------------------------------------
# Load and deduplicate annotations
# ------------------------------------------------------------

nebius_annotation_canonical = {}

raw_annotation_rows = 0
duplicate_annotation_rows = 0

for record in iter_annotation_records(
    NEBIUS_ANNOTATION_FILE
):
    raw_annotation_rows += 1

    trajectory_id = record.get("trajectory_id")
    classification = record.get("classification")

    if not trajectory_id:
        raise ValueError(
            "Annotation record is missing trajectory_id:\n"
            f"{record}"
        )

    canonical_model = canonicalize_annotation_model(
        record
    )

    instance_id = recover_instance_id(record)

    canonical_record = record.copy()
    canonical_record["_canonical_model"] = canonical_model
    canonical_record["_instance_id"] = instance_id

    canonical_key = (
        canonical_model,
        str(trajectory_id),
    )

    if canonical_key in nebius_annotation_canonical:
        existing = nebius_annotation_canonical[
            canonical_key
        ]

        if existing != canonical_record:
            raise ValueError(
                "Conflicting duplicate annotation for key:\n"
                f"{canonical_key}"
            )

        duplicate_annotation_rows += 1
        continue

    nebius_annotation_canonical[
        canonical_key
    ] = canonical_record


# ------------------------------------------------------------
# Validate annotation population
# ------------------------------------------------------------

annotation_classification_counts = pd.Series(
    [
        annotation.get("classification")
        for annotation
        in nebius_annotation_canonical.values()
    ]
).value_counts(dropna=False)

print("=== Nebius annotations ===")
print(f"Raw annotation rows: {raw_annotation_rows:,}")
print(
    "Duplicate annotation rows removed:",
    f"{duplicate_annotation_rows:,}",
)
print(
    "Canonical annotation records:",
    f"{len(nebius_annotation_canonical):,}",
)

display(
    annotation_classification_counts
    .rename_axis("classification")
    .reset_index(name="count")
)

assert len(nebius_annotation_canonical) == 9_320, (
    "Expected 9,320 final Nebius annotations, "
    f"found {len(nebius_annotation_canonical):,}."
)

assert annotation_classification_counts.get(
    TP_LABEL,
    0,
) == 9_320, (
    "Expected all 9,320 final records to be "
    "TEST_PANDERING."
)

print("✓ Final annotation population validated.")

=== Nebius annotations ===
Raw annotation rows: 9,320
Duplicate annotation rows removed: 0
Canonical annotation records: 9,320


,classification,count
0,TEST_PANDERING,9320


✓ Final annotation population validated.


In [5]:
# ============================================================
# Cell 3 — Load the raw Nebius trajectory dataset
# ============================================================

dataset = load_dataset(
    NEBIUS_DATASET,
    split=NEBIUS_SPLIT,
    streaming=True,
    cache_dir=str(HF_CACHE_DIR),
)

required_raw_columns = {
    "instance_id",
    "model_name",
    "target",
}

missing_raw_columns = (
    required_raw_columns
    - set(dataset.column_names)
)

assert not missing_raw_columns, (
    "Raw Nebius dataset is missing columns: "
    f"{sorted(missing_raw_columns)}"
)

print("=== Raw Nebius dataset ===")
print("Dataset:", NEBIUS_DATASET)
print("Split:", NEBIUS_SPLIT)
print("Columns:", dataset.column_names)
print("✓ Streaming dataset loaded.")

=== Raw Nebius dataset ===
Dataset: nebius/SWE-agent-trajectories
Split: train
Columns: ['instance_id', 'model_name', 'target', 'trajectory', 'exit_status', 'generated_patch', 'eval_logs']
✓ Streaming dataset loaded.


In [6]:
# ============================================================
# Cell 4 — Build the exact-linkage k=1 population
#
# Exclusion rule:
#   An annotation is included only when its stored canonical
#   model and trajectory ID exactly match the raw trajectory
#   identified by the zero-based global dataset index.
# ============================================================

trajectory_rows = []
matched_tp_keys = set()

raw_trajectory_count = 0
non_successful_trajectory_count = 0
unknown_model_count = 0

for global_index, raw_record in enumerate(dataset):

    raw_trajectory_count += 1

    raw_model_name = raw_record["model_name"]

    canonical_model = NEBIUS_RAW_MODEL_MAP.get(
        raw_model_name
    )

    if canonical_model is None:
        unknown_model_count += 1
        continue

    instance_id = raw_record["instance_id"]

    trajectory_id = (
        f"{instance_id}_{global_index}"
    )

    raw_key = (
        canonical_model,
        trajectory_id,
    )

    annotation = nebius_annotation_canonical.get(
        raw_key
    )

    is_exact_tp = (
        annotation is not None
        and annotation.get("classification") == TP_LABEL
    )

    if is_exact_tp:
        matched_tp_keys.add(raw_key)

    # Only non-successful trajectories belong to the k=1
    # analysis population.
    if raw_record["target"] is not False:
        continue

    non_successful_trajectory_count += 1

    trajectory_rows.append(
        {
            "model": canonical_model,
            "pr_id": instance_id,
            "trajectory_id": trajectory_id,
            "is_tp": np.int8(is_exact_tp),
        }
    )


nebius_k1_trajectories = pd.DataFrame(
    trajectory_rows
)


# ------------------------------------------------------------
# Identify annotations excluded because exact linkage failed
# ------------------------------------------------------------

all_tp_annotation_keys = {
    key
    for key, annotation
    in nebius_annotation_canonical.items()
    if annotation.get("classification") == TP_LABEL
}

excluded_tp_keys = (
    all_tp_annotation_keys
    - matched_tp_keys
)


# ------------------------------------------------------------
# Strict validation
# ------------------------------------------------------------

assert raw_trajectory_count == 80_036, (
    "Unexpected raw trajectory count: "
    f"{raw_trajectory_count:,}"
)

assert non_successful_trajectory_count == 66_647, (
    "Unexpected non-successful trajectory count: "
    f"{non_successful_trajectory_count:,}"
)

assert unknown_model_count == 0, (
    f"Found {unknown_model_count:,} records with "
    "unknown model names."
)

assert len(matched_tp_keys) == 9_294, (
    "Expected 9,294 exactly linked TP annotations, "
    f"found {len(matched_tp_keys):,}."
)

assert len(excluded_tp_keys) == 26, (
    "Expected 26 ambiguous TP annotations, "
    f"found {len(excluded_tp_keys):,}."
)

assert (
    nebius_k1_trajectories["is_tp"].sum()
    == 9_294
)


# ------------------------------------------------------------
# Exclusion report
# ------------------------------------------------------------

excluded_rows = []

for annotation_model, trajectory_id in sorted(
    excluded_tp_keys
):
    annotation = nebius_annotation_canonical[
        (
            annotation_model,
            trajectory_id,
        )
    ]

    suffix_match = re.search(
        r"_(\d+)$",
        trajectory_id,
    )

    excluded_rows.append(
        {
            "annotation_model": annotation_model,
            "trajectory_id": trajectory_id,
            "instance_id": annotation.get("_instance_id"),
            "classification": annotation.get(
                "classification"
            ),
            "parsed_global_index": (
                int(suffix_match.group(1))
                if suffix_match
                else None
            ),
            "exclusion_reason": (
                "Annotation model conflicts with raw "
                "model at the stored global index"
            ),
        }
    )

nebius_k1_exclusions = pd.DataFrame(
    excluded_rows
)


print("=== Exact-linkage population ===")
print(
    "Raw trajectories:",
    f"{raw_trajectory_count:,}",
)
print(
    "Non-successful trajectories:",
    f"{non_successful_trajectory_count:,}",
)
print(
    "Exactly linked TP trajectories:",
    f"{len(matched_tp_keys):,}",
)
print(
    "Excluded ambiguous TP annotations:",
    f"{len(excluded_tp_keys):,}",
)

display(
    nebius_k1_trajectories
    .groupby("model", as_index=False)
    .agg(
        non_successful_trajectories=(
            "trajectory_id",
            "size",
        ),
        unsuccessful_prs=(
            "pr_id",
            "nunique",
        ),
        exactly_linked_tp=(
            "is_tp",
            "sum",
        ),
    )
)

print("✓ Exact-linkage population validated.")

=== Exact-linkage population ===
Raw trajectories: 80,036
Non-successful trajectories: 66,647
Exactly linked TP trajectories: 9,294
Excluded ambiguous TP annotations: 26


,model,non_successful_trajectories,unsuccessful_prs,exactly_linked_tp
0,Llama-405B,883,98,102
1,Llama-70B,62325,3555,8840
2,Llama-8B,3439,523,352


✓ Exact-linkage population validated.


In [11]:
# ============================================================
# Cell 5 — PR-any and k=1 equal-exposure analysis
# ============================================================


# ------------------------------------------------------------
# 1. Aggregate by model and PR
# ------------------------------------------------------------

nebius_pr_exposure = (
    nebius_k1_trajectories
    .groupby(
        ["model", "pr_id"],
        as_index=False,
    )
    .agg(
        n_non_successful=(
            "trajectory_id",
            "size",
        ),
        n_tp=(
            "is_tp",
            "sum",
        ),
    )
)

assert (
    nebius_pr_exposure["n_tp"]
    <= nebius_pr_exposure["n_non_successful"]
).all()

nebius_pr_exposure["pr_any_tp"] = (
    nebius_pr_exposure["n_tp"] > 0
).astype(np.int8)


# ------------------------------------------------------------
# 2. PR-any and trajectory-level summaries
# ------------------------------------------------------------

pr_any_summary = (
    nebius_pr_exposure
    .groupby(
        "model",
        as_index=False,
    )
    .agg(
        n_prs=(
            "pr_id",
            "size",
        ),
        n_non_successful=(
            "n_non_successful",
            "sum",
        ),
        n_tp=(
            "n_tp",
            "sum",
        ),
        tp_affected_prs=(
            "pr_any_tp",
            "sum",
        ),
        median_attempts_per_pr=(
            "n_non_successful",
            "median",
        ),
    )
)

pr_any_summary["pr_any_pct"] = (
    pr_any_summary["tp_affected_prs"]
    / pr_any_summary["n_prs"]
    * 100
)

pr_any_summary["trajectory_level_pct"] = (
    pr_any_summary["n_tp"]
    / pr_any_summary["n_non_successful"]
    * 100
)


# ------------------------------------------------------------
# 3. Run 1,000 equal-exposure repetitions
# ------------------------------------------------------------
# For a PR with:
#   n_tp TP trajectories
#   n_non_successful eligible trajectories
#
# selecting one trajectory uniformly at random yields:
#
#   P(selected trajectory is TP)
#       = n_tp / n_non_successful
#
# The Bernoulli implementation below is equivalent to explicitly
# sampling one trajectory from each PR.

rng = np.random.default_rng(
    K1_RANDOM_SEED
)

replication_rows = []

for model, model_df in nebius_pr_exposure.groupby(
    "model",
    sort=True,
):

    tp_probability = (
        model_df["n_tp"].to_numpy(
            dtype=float
        )
        / model_df["n_non_successful"].to_numpy(
            dtype=float
        )
    )

    n_prs = len(tp_probability)

    sampled_tp = (
        rng.random(
            size=(
                K1_REPLICATIONS,
                n_prs,
            )
        )
        < tp_probability
    )

    prevalence_by_replication = (
        sampled_tp.mean(axis=1)
        * 100
    )

    replication_rows.extend(
        {
            "model": model,
            "replication": replication + 1,
            "k1_prevalence_pct": prevalence,
        }
        for replication, prevalence
        in enumerate(
            prevalence_by_replication
        )
    )


nebius_k1_replications = pd.DataFrame(
    replication_rows
)


# ------------------------------------------------------------
# 4. Summarize the repetition distribution
# ------------------------------------------------------------

k1_summary = (
    nebius_k1_replications
    .groupby(
        "model",
        as_index=False,
    )
    .agg(
        k1_median_pct=(
            "k1_prevalence_pct",
            "median",
        ),
        k1_p2_5_pct=(
            "k1_prevalence_pct",
            lambda values: np.percentile(
                values,
                2.5,
            ),
        ),
        k1_p97_5_pct=(
            "k1_prevalence_pct",
            lambda values: np.percentile(
                values,
                97.5,
            ),
        ),
        k1_mean_pct=(
            "k1_prevalence_pct",
            "mean",
        ),
    )
)


# ------------------------------------------------------------
# 5. Combine PR-any and k=1 results
# ------------------------------------------------------------

nebius_exposure_summary = (
    pr_any_summary
    .merge(
        k1_summary,
        on="model",
        how="inner",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# 6. Round percentage results
# ------------------------------------------------------------

percentage_columns = [
    "pr_any_pct",
    "trajectory_level_pct",
    "k1_median_pct",
    "k1_p2_5_pct",
    "k1_p97_5_pct",
    "k1_mean_pct",
]

nebius_exposure_summary[
    percentage_columns
] = (
    nebius_exposure_summary[
        percentage_columns
    ]
    .round(2)
)


# ------------------------------------------------------------
# 7. Create paper-ready percentile-range column
# ------------------------------------------------------------

nebius_exposure_summary[
    "k1_median_95_range"
] = nebius_exposure_summary.apply(
    lambda row: (
        f"{row['k1_median_pct']:.2f} "
        f"[{row['k1_p2_5_pct']:.2f}, "
        f"{row['k1_p97_5_pct']:.2f}]"
    ),
    axis=1,
)


# ------------------------------------------------------------
# 8. Apply fixed model ordering
# ------------------------------------------------------------

model_order = [
    "Llama-405B",
    "Llama-70B",
    "Llama-8B",
]

nebius_exposure_summary["model"] = pd.Categorical(
    nebius_exposure_summary["model"],
    categories=model_order,
    ordered=True,
)

nebius_exposure_summary = (
    nebius_exposure_summary
    .sort_values("model")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. Validate final results
# ------------------------------------------------------------

assert len(nebius_exposure_summary) == 3

assert len(nebius_k1_replications) == (
    len(model_order)
    * K1_REPLICATIONS
)

assert (
    nebius_k1_replications
    .groupby("model")
    .size()
    .eq(K1_REPLICATIONS)
    .all()
)

assert (
    nebius_exposure_summary["k1_p2_5_pct"]
    <= nebius_exposure_summary["k1_median_pct"]
).all()

assert (
    nebius_exposure_summary["k1_median_pct"]
    <= nebius_exposure_summary["k1_p97_5_pct"]
).all()


# ------------------------------------------------------------
# 10. Display final results
# ------------------------------------------------------------

display_columns = [
    "model",
    "n_prs",
    "n_non_successful",
    "n_tp",
    "tp_affected_prs",
    "median_attempts_per_pr",
    "pr_any_pct",
    "trajectory_level_pct",
    "k1_mean_pct",
    "k1_median_pct",
    "k1_p2_5_pct",
    "k1_p97_5_pct",
    "k1_median_95_range",
]

display(
    nebius_exposure_summary[
        display_columns
    ]
)

print(
    "✓ Nebius PR-any and k=1 analysis completed."
)

print(
    f"✓ Repetitions per model: "
    f"{K1_REPLICATIONS:,}"
)

print(
    f"✓ Total repetition records: "
    f"{len(nebius_k1_replications):,}"
)

,model,n_prs,n_non_successful,n_tp,tp_affected_prs,median_attempts_per_pr,pr_any_pct,trajectory_level_pct,k1_mean_pct,k1_median_pct,k1_p2_5_pct,k1_p97_5_pct,k1_median_95_range
0,Llama-405B,98,883,102,45,10.0,45.92,11.55,11.34,11.22,6.12,16.33,"11.22 [6.12, 16.33]"
1,Llama-70B,3555,62325,8840,2195,13.0,61.74,14.18,13.90,13.92,12.88,14.85,"13.92 [12.88, 14.85]"
2,Llama-8B,523,3439,352,192,5.0,36.71,10.24,10.87,10.90,8.60,13.19,"10.90 [8.60, 13.19]"


✓ Nebius PR-any and k=1 analysis completed.
✓ Repetitions per model: 1,000
✓ Total repetition records: 3,000
